# Omni NCM: streamed multimodal pipeline (data -> train -> val -> eval)

Level-parallel coarse-to-fine generation across **text + image + audio**, with **video** in the
data path. One schedule, one generator, K forwards per sample regardless of how many
modalities or tokens.

Data plumbing follows `DAPT_v2_stream.ipynb`: a **background producer** streams HF sources,
encodes each sample to K nested code levels with a frozen codec, writes self-describing `.npz`
shards into a rolling buffer, holds out eval samples per group, and signals readiness with a
`EVAL_READY` file. Training reads shards and starts about a minute in.

**Code runs from local disk, not Drive.** Drive IO stalls the producer and every shard read;
only the final checkpoints are copied back.

| Cell | What it does |
|---|---|
| 1 | Mount Drive, stage the package to `/content`, install `datasets`/`soundfile`/`av` |
| 2 | `run_omni.py check` - 7 model invariants + all four codecs + producer selfcheck, no data |
| 3 | Launch the streaming producer, wait for held-out blocks, keep the handle |
| 4 | Smoke run: 10 steps, s/step, projected hours **and dollars** |
| 5 | Train the level-parallel arm |
| 6 | Train the matched autoregressive arm (identical params, causal mask) |
| 7 | Eval: quality vs NFE table, both arms |
| 8 | Copy checkpoints and the eval table back to Drive |

**The claim under test:** level-parallel reaches the autoregressive arm's quality at NFE = K
instead of NFE = total tokens. If cell 7 shows AR matching quality at NFE <= K, the thesis is
dead and the write-up is the negative result.

Upload either `ncm_omni.zip` or the `ncm_omni/` folder to `MyDrive/ncm_omni_pkg/` before cell 1.
The package must include `core.py` (the kernel-optimised NCM core) at its root.

In [ ]:
import os, sys, shutil, subprocess, zipfile

DRIVE = '/content/drive/MyDrive/ncm_omni_pkg'
PKG   = '/content/ncm_omni'          # working copy: local disk, always

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(PKG):
    zip_path = os.path.join(DRIVE, 'ncm_omni.zip')
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall('/content')
        if not os.path.isdir(PKG) and os.path.isdir('/content/ncm_omni_pkg'):
            shutil.move('/content/ncm_omni_pkg', PKG)
    elif os.path.isdir(os.path.join(DRIVE, 'ncm_omni')):
        shutil.copytree(os.path.join(DRIVE, 'ncm_omni'), PKG)

assert os.path.isdir(PKG), f'stage the package at {DRIVE} (zip or folder) and re-run this cell'
assert os.path.exists(os.path.join(PKG, 'core.py')), 'core.py must sit at the package root'
os.chdir(PKG)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                       'datasets', 'soundfile', 'av'])

import torch
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('workdir', os.getcwd())
print(sorted(os.listdir(PKG)))


In [ ]:
# Invariants first: level k cannot see level k+1, the cross-level knob is wired, every modality
# and level gets a gradient, an absent modality is masked out, AR mode does not leak the future,
# generation costs exactly K forwards, and every codec emits the exact schedule ladder.
!python run_omni.py check


In [ ]:
import time

subprocess.check_call([sys.executable, 'run_omni.py', 'cfg'])          # writes run/cfg/*.json
STREAM = os.path.join(PKG, 'run', 'stream')
LOG    = os.path.join(PKG, 'run', 'producer.log')
os.makedirs(os.path.dirname(LOG), exist_ok=True)

PRODUCER = subprocess.Popen(
    [sys.executable, '-m', 'omni.producer', os.path.join(PKG, 'run', 'cfg', 'produce.json')],
    stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)
print('producer pid', PRODUCER.pid, '- it keeps running after this cell')

t0 = time.time()
while not os.path.exists(os.path.join(STREAM, 'EVAL_READY')):
    if PRODUCER.poll() is not None:
        raise RuntimeError('producer died early:\n' + ''.join(open(LOG).readlines()[-30:]))
    if time.time() - t0 > 1800:
        print(''.join(open(LOG).readlines()[-12:]))
        raise TimeoutError('no held-out blocks in 30 min: check which sources opened')
    time.sleep(5)
print(f'held-out blocks ready after {time.time()-t0:.0f}s')
print(''.join(open(LOG).readlines()[-10:]))


In [ ]:
# 10 steps in, 10 steps out. Prints s/step plus the projected wall clock and cost for the
# configured step budget, so the real run is never a surprise.
!python run_omni.py train --smoke 10


In [ ]:
# Arm 1: level-parallel. Val accuracy is printed per (modality, level) every eval_every steps.
!python run_omni.py train


In [ ]:
# Arm 2: autoregressive, same width/depth/schedule, causal mask over the same raster.
# Same shards, same steps: matched params, matched tokens, one variable changed.
!python run_omni.py train --arm ar


In [ ]:
# The only table that matters: quality vs NFE, greedy, teacher-free, held-out samples.
!python run_omni.py eval


In [ ]:
# Persist: checkpoints and the table are the only things worth keeping off local disk.
import glob, os, shutil

DST = '/content/drive/MyDrive/ncm_omni_runs'
os.makedirs(DST, exist_ok=True)
for src in glob.glob(os.path.join(PKG, 'run', 'train_*', '*.pt')) + \
           glob.glob(os.path.join(PKG, 'run', '*.json')):
    shutil.copy2(src, os.path.join(DST, os.path.basename(src)))
print('copied', len(os.listdir(DST)), 'files to', DST)
print(sorted(os.listdir(DST))[:12])


## Reading cell 7

`NFE` is the number of network evaluations used to produce one sample: the number of levels for
the parallel arm, the number of tokens for the AR arm. A packed sample here is ~506 positions,
so the AR arm's NFE is in the hundreds - that is the whole point of the comparison.

* **Thesis holds** if the parallel arm's `mean acc` at `NFE = K` meets or beats AR's plateau,
  in every modality. Report the ratio.
* **Partial win** is the expected outcome: text and image hold, audio's fine level sags, because
  pooling a spectrogram by 32x destroys the formants. That is a real finding and it is
  publishable - say so instead of hiding it.
* **Thesis dies** if AR reaches the same accuracy at `NFE <= K`, or if the parallel arm's fine
  levels are flat while coarse levels improve (the failure mode `cross_level` exists to test).

Before any of this means anything, the codecs must be real. Today they are: byte codec for text
(finest level decodes exactly), patch-mean pyramid for image and video, log-mel rate pyramid for
audio over a **fixed seeded codebook**. The ladder geometry is the real claim; the codebooks are
what a trained VQ (NCM's text RVQ, Infinity's image VQ, DAC for audio) replaces, and
`omni/codecs.py::build_codec` is the single place to swap them.

Codes are 1-based and 0 is NULL, so `vocab = codebook_size + 1`. Do not "fix" that back to
0-based without also changing `null_code`: real codes would be masked out of the loss silently.